# 04 – Transformer Model: DistilBERT

## Overview

### What is DistilBERT?

**DistilBERT** (Distilled BERT) is a lighter, faster version of BERT (Bidirectional Encoder Representations from Transformers) developed by Hugging Face via **knowledge distillation** — a technique where a smaller "student" model learns to mimic a larger "teacher" model.

| Property | BERT-base | DistilBERT |
|---|---|---|
| Parameters | 110M | 66M (40% smaller) |
| Layers | 12 | 6 |
| Hidden size | 768 | 768 |
| Inference speed | Baseline | ~60% faster |
| Performance | 100% | ~97% of BERT |

### Why DistilBERT for News Classification?

1. **Contextual understanding**: Unlike TF-IDF which treats words as independent features, DistilBERT understands context. "Apple releases new product" vs "Apple fell from the tree" — DistilBERT knows they differ.
2. **Pre-trained on massive corpora**: Trained on English Wikipedia + BookCorpus (3.3B words), it brings rich linguistic knowledge.
3. **Fine-tuning efficiency**: Only needs a small labelled dataset (AG News has 120K examples — more than enough).
4. **Subword tokenization**: Handles out-of-vocabulary words (e.g., new tech jargon) gracefully via WordPiece tokenization.
5. **Practical size**: At 66M parameters (~250MB), it's deployable on standard hardware.

The AG News dataset has 4 well-separated categories (World, Sports, Business, Sci/Tech) but some overlap (e.g., tech companies appear in both Business and Sci/Tech). DistilBERT's contextual embeddings are better suited to resolve such ambiguity than bag-of-words TF-IDF features.

In [ ]:
# Setup — import required libraries
import importlib
import sys

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

print(f"Python  : {sys.version.split()[0]}")
print(f"NumPy   : {np.__version__}")
print(f"Pandas  : {pd.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")

# Optional heavy libraries — check availability
try:
    import torch
    print(f"PyTorch : {torch.__version__}")
    print(f"  CUDA available : {torch.cuda.is_available()}")
    TORCH_AVAILABLE = True
except ImportError:
    print("PyTorch : NOT INSTALLED  (pip install torch)")
    TORCH_AVAILABLE = False

try:
    import transformers
    print(f"Transformers: {transformers.__version__}")
    TRANSFORMERS_AVAILABLE = True
except ImportError:
    print("Transformers: NOT INSTALLED  (pip install transformers)")
    TRANSFORMERS_AVAILABLE = False

print("\nSetup complete.")

In [ ]:
# Tokenization Demo — WordPiece vs. simple whitespace splitting

SAMPLE_HEADLINE = (
    "NASA's Perseverance rover discovers biosignatures on Mars, "
    "scientists say findings are extraordinary"
)

print("=" * 70)
print("SAMPLE HEADLINE")
print("=" * 70)
print(SAMPLE_HEADLINE)
print()

# ── Simple whitespace tokenization (baseline) ──────────────────────────
simple_tokens = SAMPLE_HEADLINE.split()
print(f"Simple (whitespace) tokenization — {len(simple_tokens)} tokens:")
print(simple_tokens)
print()

# ── DistilBERT WordPiece tokenization ─────────────────────────────────
if not TRANSFORMERS_AVAILABLE:
    print(
        "[SKIPPED] 'transformers' library is not installed.\n"
        "Install with: pip install transformers\n"
        "The demo below shows what the output would look like."
    )
    # Illustrative mock output so the notebook is still educational
    mock_tokens = [
        '[CLS]', 'nasa', "'", 's', 'perseverance', 'rover', 'discovers',
        'bio', '##sign', '##atures', 'on', 'mars', ',', 'scientists', 'say',
        'findings', 'are', 'extraordinary', '[SEP]'
    ]
    print(f"\n[MOCK] DistilBERT WordPiece tokens — {len(mock_tokens)} tokens:")
    print(mock_tokens)
    print("\nNotice:")
    print("  • 'biosignatures' is split into ['bio', '##sign', '##atures']")
    print("  • Special tokens [CLS] (start) and [SEP] (end) are added automatically")
    print("  • Tokens starting with '##' are word-continuations (subwords)")
else:
    from transformers import DistilBertTokenizer

    print("Loading distilbert-base-uncased tokenizer …")
    tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

    encoding = tokenizer(
        SAMPLE_HEADLINE,
        return_tensors="pt",
        max_length=128,
        padding="max_length",
        truncation=True,
    )

    # Remove padding for display
    ids_no_pad = [i for i in encoding["input_ids"][0].tolist() if i != 0]
    mask_no_pad = [m for m in encoding["attention_mask"][0].tolist() if m != 0]
    tokens = tokenizer.convert_ids_to_tokens(ids_no_pad)

    print(f"\nDistilBERT WordPiece tokens — {len(tokens)} tokens (before padding):")
    print(tokens)

    print(f"\ninput_ids (first 20):    {ids_no_pad[:20]}")
    print(f"attention_mask (first 20): {mask_no_pad[:20]}")

    print("\nKey observations:")
    print(f"  • Simple split  : {len(simple_tokens):>3} tokens")
    print(f"  • WordPiece     : {len(tokens):>3} tokens (incl. [CLS] & [SEP])")
    print("  • Subwords (starting with ##) handle rare/compound words")
    print("  • [CLS] token carries the sentence-level representation used for classification")
    print("  • [SEP] token marks sequence boundaries")
    print("  • Padding fills positions up to max_length=128 (attention_mask=0 for padding)")

## DistilBERT Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                    INPUT TEXT (news article)                    │
└───────────────────────────┬─────────────────────────────────────┘
                            │
                            ▼
┌─────────────────────────────────────────────────────────────────┐
│                 TOKENIZER (WordPiece)                           │
│  "NASA's rover" → [CLS, nasa, ', s, rover, SEP, PAD, PAD ...]  │
└───────────────────────────┬─────────────────────────────────────┘
                            │  token IDs + attention mask
                            ▼
┌─────────────────────────────────────────────────────────────────┐
│              EMBEDDING LAYER  (vocab_size=30,522, dim=768)      │
│         Token Embeddings + Positional Embeddings                │
└───────────────────────────┬─────────────────────────────────────┘
                            │  [batch, seq_len, 768]
                            ▼
┌─────────────────────────────────────────────────────────────────┐
│                  TRANSFORMER BLOCK × 6                          │
│  ┌──────────────────────────────────────────────────────────┐   │
│  │  Multi-Head Self-Attention (12 heads)                    │   │
│  │    Each head learns: Q=query, K=key, V=value matrices    │   │
│  │    Attention(Q,K,V) = softmax(QKᵀ / √d_k) · V           │   │
│  │    Allows every token to attend to every other token     │   │
│  ├──────────────────────────────────────────────────────────┤   │
│  │  Add & Layer Norm                                        │   │
│  ├──────────────────────────────────────────────────────────┤   │
│  │  Feed-Forward Network  (768 → 3072 → 768)                │   │
│  ├──────────────────────────────────────────────────────────┤   │
│  │  Add & Layer Norm                                        │   │
│  └──────────────────────────────────────────────────────────┘   │
│                         × 6 layers                              │
└───────────────────────────┬─────────────────────────────────────┘
                            │  [batch, seq_len, 768]
                            │  Take [CLS] token output → [batch, 768]
                            ▼
┌─────────────────────────────────────────────────────────────────┐
│              CLASSIFICATION HEAD                                │
│         Linear(768 → 4)  →  Softmax  →  4 class probs          │
└─────────────────────────────────────────────────────────────────┘
```

### How Self-Attention Works (Intuition)

Self-attention allows each token to look at every other token in the sequence and decide how much to "attend" to each one when building its representation.

**Example**: In *"Apple stock fell after the iPhone announcement"*:
- The word **"Apple"** attends strongly to **"stock"**, **"fell"**, **"announcement"** → context tells the model this is about the company, not the fruit.
- TF-IDF would treat "Apple" identically in both cases.

**Multi-head attention** runs 12 independent attention operations in parallel, each learning different relationship patterns (syntax, coreference, semantics), then concatenates the results.

### The [CLS] Token

A special `[CLS]` (classification) token is prepended to every input. After passing through all 6 transformer blocks, the `[CLS]` token's 768-dimensional hidden state has absorbed contextual information from the entire sequence. The classification head uses only this vector to predict the news category.

In [ ]:
# Model configuration — training hyperparameters (display only, no training)
import json

config = {
    "model_name": "distilbert-base-uncased",
    "max_length": 128,          # max token sequence length
    "batch_size": 16,           # mini-batch size for gradient updates
    "learning_rate": 2e-5,      # small LR essential for fine-tuning pre-trained weights
    "epochs": 3,                # 3 epochs is standard for BERT-style fine-tuning
    "optimizer": "AdamW",       # Adam with decoupled weight decay
    "weight_decay": 0.01,       # L2 regularisation (applied via optimizer, not loss)
    "warmup_ratio": 0.1,        # first 10% of steps used for LR warm-up
    "num_classes": 4,           # World, Sports, Business, Sci/Tech
}

print("DistilBERT Fine-Tuning Configuration")
print("=" * 45)
for key, value in config.items():
    print(f"  {key:<20}: {value}")

# Dataset split info
print("\nDataset Split")
print("=" * 45)
dataset_info = {
    "train_samples"   : 120_000,
    "test_samples"    : 7_600,
    "classes"         : ["World", "Sports", "Business", "Sci/Tech"],
    "samples_per_class": "30,000 (train) / 1,900 (test)",
}
for key, value in dataset_info.items():
    print(f"  {key:<22}: {value}")

# Compute rough training-time estimate
steps_per_epoch = dataset_info["train_samples"] // config["batch_size"]
total_steps = steps_per_epoch * config["epochs"]
warmup_steps = int(total_steps * config["warmup_ratio"])

print("\nDerived Training Parameters")
print("=" * 45)
print(f"  steps_per_epoch : {steps_per_epoch:,}")
print(f"  total_steps     : {total_steps:,}")
print(f"  warmup_steps    : {warmup_steps:,}")

In [ ]:
# Training procedure — pseudocode walkthrough
# This cell does NOT execute training; it explains the training loop.

TRAINING_PSEUDOCODE = """
# ─── DistilBERT Fine-Tuning Loop (Pseudocode) ───────────────────────────

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=4
)

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,    # gradual LR ramp-up
    num_training_steps=total_steps
)

for epoch in range(3):
    model.train()
    for batch in train_dataloader:         # batches of 16 examples
        input_ids      = batch["input_ids"]        # [16, 128]
        attention_mask = batch["attention_mask"]   # [16, 128]
        labels         = batch["labels"]           # [16]

        # Forward pass
        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels)
        loss = outputs.loss                # cross-entropy loss

        # Backward pass
        loss.backward()                    # compute gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # prevent exploding gradients
        optimizer.step()                   # update weights
        scheduler.step()                   # update learning rate
        optimizer.zero_grad()

    # Evaluation after each epoch
    model.eval()
    evaluate(model, val_dataloader)

# Save fine-tuned model
model.save_pretrained("../models/transformer/distilbert_agnews")
tokenizer.save_pretrained("../models/transformer/distilbert_agnews")
"""

print(TRAINING_PSEUDOCODE)

print("="*70)
print("WHY THESE DESIGN CHOICES?")
print("="*70)
print("""
AdamW (not plain Adam):
  Adam's built-in L2 regularisation is applied to the adapted learning rate,
  which is sub-optimal. AdamW decouples weight decay from the gradient update,
  giving cleaner regularisation — critical for fine-tuning large pre-trained models.

Small Learning Rate (2e-5):
  Pre-trained weights encode millions of gradient updates on large corpora.
  A large LR would 'catastrophically forget' this knowledge. 2e-5 is a well-
  established default that gently adapts the weights to the new task.

Linear LR Warm-up:
  The first 10% of training steps ramp the LR from ~0 to 2e-5 linearly.
  This prevents the model's sensitive early gradients from making oversized
  updates before the optimiser has stable momentum estimates.

Gradient Clipping (max_norm=1.0):
  Caps gradient magnitudes to prevent rare but catastrophic 'gradient explosions',
  which are especially common when dealing with long sequences.

3 Epochs:
  BERT-style models generalise quickly to downstream tasks. More than 3-5 epochs
  typically causes overfitting on classification tasks with 100K+ samples.
""")

## Expected Performance Estimates

The table below compares our trained **TF-IDF + Logistic Regression** baseline against the **expected** performance of a fine-tuned DistilBERT model on the AG News dataset, based on published research and Hugging Face benchmark results.

| Metric | TF-IDF + LR | DistilBERT (expected) |
|---|---|---|
| Accuracy | 91.76% | ~94–95% |
| Macro F1 | 91.75% | ~94–95% |
| Training time | ~12 seconds | ~2–3 hours (CPU) / ~20 min (GPU) |
| Model size on disk | ~15 MB | ~250 MB |
| Inference latency | <1 ms/sample | ~50 ms/sample (CPU) |
| Memory footprint | <100 MB | ~1 GB (inference) |

> **Note**: These DistilBERT estimates are based on the [Hugging Face AG News benchmark](https://huggingface.co/datasets/ag_news) and papers such as Sanh et al. (2019) *DistilBERT, a distilled version of BERT*.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Visualise the expected performance comparison
metrics = ["Accuracy", "Macro F1"]
tfidf_scores = [91.76, 91.75]
distilbert_expected = [94.5, 94.5]   # midpoint of 94–95%

x = np.arange(len(metrics))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Bar chart ──────────────────────────────────────────────────────────
ax = axes[0]
bars1 = ax.bar(x - width/2, tfidf_scores,  width, label="TF-IDF + LR (actual)",      color="#4C72B0", alpha=0.85)
bars2 = ax.bar(x + width/2, distilbert_expected, width, label="DistilBERT (expected)", color="#DD8452", alpha=0.85, hatch="//")

ax.set_ylim(88, 97)
ax.set_ylabel("Score (%)")
ax.set_title("Model Performance Comparison", fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.set_axisbelow(True)
ax.yaxis.grid(True, linestyle="--", alpha=0.7)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f"{bar.get_height():.2f}%", ha="center", va="bottom", fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f"~{bar.get_height():.1f}%", ha="center", va="bottom", fontsize=9, style="italic")

# ── Training time comparison (log scale) ──────────────────────────────
ax2 = axes[1]
models = ["TF-IDF + LR", "DistilBERT\n(CPU)", "DistilBERT\n(GPU)"]
times_seconds = [12, 2.5 * 3600, 20 * 60]   # 12s, 2.5h, 20min
colors = ["#4C72B0", "#DD8452", "#55A868"]

bars3 = ax2.bar(models, times_seconds, color=colors, alpha=0.85)
ax2.set_yscale("log")
ax2.set_ylabel("Training Time (seconds, log scale)")
ax2.set_title("Training Time Comparison", fontweight="bold")
ax2.set_axisbelow(True)
ax2.yaxis.grid(True, linestyle="--", alpha=0.7)

labels_time = ["12 sec", "~2.5 hrs", "~20 min"]
for bar, label in zip(bars3, labels_time):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.3,
             label, ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.show()

## How to Train the DistilBERT Model

The training script is located at `scripts/train_transformer.py`. Once the dependencies are installed, run:

```bash
# Install dependencies first (one-time)
pip install torch transformers datasets scikit-learn

# Run training
python scripts/train_transformer.py
```

### Expected Console Output

```
Loading dataset from data/processed/ ...
  Train: 120,000 samples | Test: 7,600 samples

Loading distilbert-base-uncased tokenizer ...
Tokenising train set ... done in 142s
Tokenising test  set ... done in 9s

Loading DistilBertForSequenceClassification (num_labels=4) ...
Total parameters : 66,955,780
Trainable params : 66,955,780

Epoch 1/3  [████████████████████] 7500/7500 — loss: 0.2341 — val_acc: 0.9312
Epoch 2/3  [████████████████████] 7500/7500 — loss: 0.1198 — val_acc: 0.9421
Epoch 3/3  [████████████████████] 7500/7500 — loss: 0.0876 — val_acc: 0.9463

Final Test Accuracy : 94.63%
Final Macro F1      : 94.58%

Model saved to models/transformer/distilbert_agnews/
```

### Hardware Requirements

| Requirement | Minimum | Recommended |
|---|---|---|
| RAM | 8 GB | 16 GB |
| GPU VRAM | — (CPU mode) | 4 GB+ (CUDA) |
| Storage | 300 MB free | 1 GB free |
| Training time | ~2–3 hours (CPU) | ~20 minutes (GPU) |

> **Tip**: If you have a Google account, you can train for free on **Google Colab** with a T4 GPU. Upload the `data/processed/` directory and `scripts/train_transformer.py`, then run the script — training should complete in under 25 minutes.

## TF-IDF vs. Transformer: Deep Comparison

| Dimension | TF-IDF + Logistic Regression | DistilBERT (Transformer) |
|---|---|---|
| **Representation** | Sparse bag-of-words (word frequency) | Dense contextual embeddings (meaning-aware) |
| **Interpretability** | ✅ High — inspect feature weights directly | ⚠️ Partial — attention visualisation, SHAP/LIME needed |
| **Training data needs** | Works well with 10K+ samples | Needs 1K+ (fine-tuning); pre-training = 3.3B words |
| **Computational cost** | ✅ Very low (seconds on CPU) | ❌ High (hours on CPU, minutes on GPU) |
| **Inference speed** | ✅ <1 ms/sample | ❌ ~50 ms/sample (CPU) |
| **Out-of-vocabulary** | ❌ Unknown words → ignored | ✅ WordPiece handles unseen words via subwords |
| **Context sensitivity** | ❌ "Apple" same in all contexts | ✅ Disambiguates by surrounding words |
| **Domain adaptation** | ❌ Requires re-vectorisation | ✅ Lightweight fine-tuning on new domain |
| **Multi-lingual** | ❌ Language-specific features | ✅ Multilingual models (mDistilBERT) available |
| **Max sequence length** | Unlimited (memory permitting) | 512 tokens max (standard) |
| **Model size** | ~15 MB | ~250 MB |
| **Deployment complexity** | ✅ Simple (sklearn pickle) | ⚠️ Requires transformers library |

## When to Use Each Approach

### Choose TF-IDF + Classical ML when:

- ⚡ **Speed is critical**: You need predictions in microseconds (real-time systems, low-latency APIs)
- 💾 **Resources are constrained**: Embedded systems, edge devices, or budget cloud instances
- 🔍 **Interpretability is required**: Regulatory/compliance contexts where decisions must be explainable
- 📊 **Small dataset**: Under 1,000 labelled examples — transformers overfit, classical ML generalises better
- 🚀 **Rapid prototyping**: You need a working baseline in hours, not days
- 📰 **Simple classification**: Topics are well-separated with distinct vocabulary (like AG News Sports)

### Choose DistilBERT / Transformers when:

- 🧠 **Semantics matter**: Sarcasm, negation, context-dependent meaning
- 🌐 **Cross-domain**: You need the model to generalise to new writing styles/domains
- 🔤 **Noisy text**: User-generated content, social media, abbreviations
- 📈 **Squeezing accuracy**: Every percentage point matters (e.g., medical diagnosis, financial news)
- 🔁 **Transfer learning**: You want to leverage pre-training on massive corpora
- 🌍 **Multi-lingual**: News from multiple languages

### Our Recommendation for AG News

The **TF-IDF + LR baseline at 91.76%** is already production-ready for many use cases. If the 2–3% accuracy gap from DistilBERT is worth the 100× increase in inference latency and 17× larger model size depends entirely on your deployment constraints.

## Summary

### Key Takeaways

1. **DistilBERT is a distilled version of BERT** — 40% smaller, 60% faster, retaining ~97% of performance. It achieves this via knowledge distillation, not architectural shortcuts.

2. **WordPiece tokenization** handles out-of-vocabulary words by splitting them into known subword units (e.g., `biosignatures` → `bio + ##sign + ##atures`). This is a fundamental advantage over simple word-based TF-IDF features.

3. **Self-attention** is the core innovation: every token can attend to every other token, enabling the model to understand context. This resolves ambiguity that TF-IDF fundamentally cannot handle.

4. **Fine-tuning requires only 3 epochs** and a small learning rate (2e-5) with AdamW + warm-up scheduling. The pre-trained weights do most of the heavy lifting.

5. **Expected accuracy of ~94–95%** on AG News — roughly 2–3% better than our TF-IDF baseline — at the cost of ~900× longer training and ~50× slower inference.

6. **The right tool depends on constraints**: TF-IDF + LR is better for speed, interpretability, and resource-constrained environments. DistilBERT wins on accuracy, robustness, and cross-domain generalisation.

7. **Next notebook** (`05_result_analysis.ipynb`) performs a deep-dive analysis of the TF-IDF + LR model's results, including error analysis, feature importance, and confidence distributions.

---
*References*:
- Sanh, V. et al. (2019). *DistilBERT, a distilled version of BERT: smaller, faster, cheaper and lighter*. arXiv:1910.01108
- Devlin, J. et al. (2018). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*. arXiv:1810.04805
- Zhang, X. et al. (2015). *Character-level Convolutional Networks for Text Classification* (AG News dataset). NeurIPS.